# 6. Interactive Map — Folium

Visualize launch sites and analyse geographic proximity.

In [ ]:
import folium
import pandas as pd
import numpy as np
from folium.plugins import MarkerCluster
from folium.plugins import MousePosition
from folium.features import DivIcon
import math

# Marker colour helper
def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'green'
    else:
        return 'red'


In [ ]:
df = pd.read_csv('../data/dataset_part_2.csv')
print(df.shape)

# Unique launch sites with lat/lon
launch_sites = df.groupby('LaunchSite')[['Latitude','Longitude']].mean().reset_index()
print(launch_sites)


## 6.1 Base Map with Site Markers

In [ ]:
# Centre on USA
site_map = folium.Map(location=[29.5, -80.0], zoom_start=5)

for _, row in launch_sites.iterrows():
    folium.Circle(
        location=[row['Latitude'], row['Longitude']],
        radius=1000, color='#d35400', fill=True,
        tooltip=row['LaunchSite']
    ).add_to(site_map)
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        icon=DivIcon(
            icon_size=(250,36),
            icon_anchor=(0,0),
            html=f'<div style="font-size:12px;color:#d35400;"><b>{row["LaunchSite"]}</b></div>'
        )
    ).add_to(site_map)

site_map


## 6.2 Launch Records with Success/Failure Colors

In [ ]:
marker_cluster = MarkerCluster()

site_map2 = folium.Map(location=[29.5, -80.0], zoom_start=5)
site_map2.add_child(marker_cluster)

for _, row in df.iterrows():
    color = assign_marker_color(row['Class'])
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        icon=folium.Icon(color=color, icon='info-sign'),
        popup=folium.Popup(
            f"<b>{row['LaunchSite']}</b><br>"
            f"Booster: {row['BoosterVersion']}<br>"
            f"Payload: {row['PayloadMass']} kg<br>"
            f"Outcome: {'Success' if row['Class']==1 else 'Failure'}",
            max_width=200
        )
    ).add_to(marker_cluster)

site_map2


## 6.3 Proximity Analysis — Distance to Coastline, Highway, City

In [ ]:
def calculate_distance(lat1, lon1, lat2, lon2):
    """Haversine distance in km."""
    R = 6371
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

# Example: KSC LC-39A proximity
ksc_lat, ksc_lon = 28.5733, -80.6469

coast_lat,    coast_lon    = 28.5618, -80.5674  # nearest coastline
highway_lat,  highway_lon  = 28.5721, -80.6570  # US-1 highway
city_lat,     city_lon     = 28.3861, -80.6078  # Cocoa, FL

print(f'KSC to coast:   {calculate_distance(ksc_lat,ksc_lon,coast_lat,coast_lon):.2f} km')
print(f'KSC to highway: {calculate_distance(ksc_lat,ksc_lon,highway_lat,highway_lon):.2f} km')
print(f'KSC to city:    {calculate_distance(ksc_lat,ksc_lon,city_lat,city_lon):.2f} km')


In [ ]:
# Draw proximity lines on map
site_map3 = folium.Map(location=[ksc_lat, ksc_lon], zoom_start=10)

folium.Marker([ksc_lat, ksc_lon], popup='KSC LC-39A',
              icon=folium.Icon(color='blue')).add_to(site_map3)

for name, lat, lon, col in [
    ('Coastline', coast_lat, coast_lon, 'blue'),
    ('Highway',   highway_lat, highway_lon, 'orange'),
    ('City',      city_lat, city_lon, 'purple'),
]:
    folium.PolyLine([[ksc_lat,ksc_lon],[lat,lon]], color=col, weight=2).add_to(site_map3)
    d = calculate_distance(ksc_lat, ksc_lon, lat, lon)
    folium.Marker([lat, lon],
        popup=f'{name}: {d:.2f} km',
        icon=DivIcon(icon_size=(150,20), icon_anchor=(0,0),
                     html=f'<div style="font-size:11px;color:{col};"><b>{name} {d:.1f}km</b></div>')
    ).add_to(site_map3)

site_map3
